# Stage 02 · Step 2 — Supervised RUL fine-tuning

Mount the SSL-pretrained encoder, attach a small regression head, and predict per-component remaining useful life (`rul_C1..rul_C6`) from a 30-day window of telemetry.

Three regimes are trained back-to-back (a 2x2 with one degenerate cell dropped — random init + frozen encoder carries no signal so we skip it):

1. **`scratch_all`** — random init, train all weights. The genuine no-SSL baseline.
2. **`pretrained_all`** — load SSL weights, train all (encoder + head). Upper bound on what SSL transfer can give us.
3. **`pretrained_frozen`** — load SSL weights, freeze encoder, train head only. Isolates the pretext-task transfer signal.

The clean SSL claim is `pretrained_frozen` beating `scratch_all` (same head architecture, same training regime, only init differs).

Validation: sliding-cumulative time-series CV inside printers 70..84 (`ml_models.lib.splits.expanding_window_folds`). Final metrics on test printers 85..99 (held out throughout).

In [1]:
from __future__ import annotations
import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import PatchTSTConfig, PatchTSTForRegression

from ml_models.lib.data import (
    DEFAULT_FLEET_PATH,
    TEST_PRINTERS,
    TRAIN_PRINTERS,
    VAL_PRINTERS,
    filter_printers,
    load_fleet,
)
from ml_models.lib.features import build_feature_matrix
from ml_models.lib.splits import expanding_window_folds
from ml_models import PROJECT_ROOT
from sdg.schema import COMPONENT_IDS

MODELS_DIR = PROJECT_ROOT / 'ml_models/02_ssl/models'
RESULTS_DIR = PROJECT_ROOT / 'ml_models/02_ssl/results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [2]:
fleet = load_fleet(DEFAULT_FLEET_PATH)
feat_fleet, feature_cols = build_feature_matrix(fleet)
scaler = np.load(MODELS_DIR / 'feature_scaler.npz', allow_pickle=True)
channel_mean = scaler['mean']; channel_std = scaler['std']
with (MODELS_DIR / 'ssl_config.json').open() as handle:
    saved = json.load(handle)
patch_cfg_dict = saved['patch_cfg']
train_cfg = saved['train_cfg']
CONTEXT_LENGTH = int(train_cfg['context_length'])
RUL_COLS = [f'rul_{component_id}' for component_id in COMPONENT_IDS]
feat_fleet[RUL_COLS] = fleet[RUL_COLS].astype('float32').to_numpy()
feat_fleet.head(3)

,printer_id,city,climate_zone,date,day,ambient_temp_c,humidity_pct,dust_concentration,Q_demand,jobs_today,...,rul_C2,rul_C3,rul_C4,rul_C5,rul_C6,rul_system,sin_doy,cos_doy,sin_month,cos_month
0,0,Helsinki,nordic,2015-01-01,0,24.830734,59.144363,50.000000,1.0,1,...,430.0,55.0,75.0,1302.0,NaN,15,0.017202,0.999852,0.5,0.866025
1,0,Helsinki,nordic,2015-01-02,1,24.840237,59.261456,50.347904,1.0,1,...,429.0,54.0,74.0,1301.0,NaN,14,0.034398,0.999408,0.5,0.866025
2,0,Helsinki,nordic,2015-01-03,2,24.849047,59.377289,51.015594,1.0,1,...,428.0,53.0,73.0,1300.0,NaN,13,0.051584,0.998669,0.5,0.866025


In [3]:
RUL_CLIP = 365.0  # cap RUL at 1 year for stable regression

class RULDataset(Dataset):
    def __init__(self, df: pd.DataFrame, printer_ids, day_range: range,
                 feature_cols: list[str], rul_cols: list[str],
                 context_length: int, mean: np.ndarray, std: np.ndarray,
                 stride: int = 14):
        self.context_length = context_length
        self.stride = int(stride)
        self.feature_cols = feature_cols
        self.mean = mean.astype(np.float32)
        self.std = std.astype(np.float32)
        keep = filter_printers(df, printer_ids)
        keep = keep[(keep['day'] >= day_range.start) & (keep['day'] < day_range.stop)]
        self.samples: list[tuple[np.ndarray, np.ndarray]] = []
        for printer_id, group in keep.groupby('printer_id', sort=False):
            arr = group[feature_cols].to_numpy(dtype=np.float32)
            ruls = group[rul_cols].to_numpy(dtype=np.float32)
            T = arr.shape[0]
            if T < context_length:
                continue
            for end in range(context_length, T, self.stride):
                window = arr[end - context_length:end]
                target = ruls[end - 1]
                if np.isnan(target).all():
                    continue
                self.samples.append((window, target))

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        window, target = self.samples[idx]
        normed = (window - self.mean) / self.std
        clipped = np.minimum(np.where(np.isnan(target), RUL_CLIP, target), RUL_CLIP)
        mask = (~np.isnan(target)).astype(np.float32)
        return (
            torch.from_numpy(normed),
            torch.from_numpy(clipped.astype(np.float32) / RUL_CLIP),
            torch.from_numpy(mask),
        )

In [4]:
def build_regression_model(load_pretrained: bool) -> PatchTSTForRegression:
    patch_cfg = PatchTSTConfig(**patch_cfg_dict)
    patch_cfg.num_targets = len(RUL_COLS)
    patch_cfg.prediction_length = 1  # we predict 6 RULs per window, not a sequence
    patch_cfg.use_cls_token = False
    model = PatchTSTForRegression(patch_cfg)
    if load_pretrained:
        state = torch.load(MODELS_DIR / 'ssl_encoder.pt', map_location='cpu')
        encoder_state = {k: v for k, v in state.items() if k.startswith('model.')}
        missing, unexpected = model.load_state_dict(encoder_state, strict=False)
        print(f'load_pretrained={load_pretrained}: missing={len(missing)} unexpected={len(unexpected)}')
    return model.to(device)

In [5]:
@dataclass
class FineTuneCfg:
    epochs: int = 3
    batch_size: int = 128
    lr_head: float = 5e-4
    lr_backbone: float = 5e-5
    weight_decay: float = 1e-4
    grad_clip: float = 1.0

ft = FineTuneCfg()

def train_one_fold(model: nn.Module, train_loader: DataLoader, freeze_encoder: bool) -> nn.Module:
    if freeze_encoder:
        for name, param in model.named_parameters():
            if not name.startswith('regression_head') and 'head' not in name:
                param.requires_grad = False
    params = [p for p in model.parameters() if p.requires_grad]
    optim = torch.optim.AdamW(params, lr=ft.lr_head, weight_decay=ft.weight_decay)
    for epoch in range(ft.epochs):
        model.train()
        running = 0.0
        for x, y, m in train_loader:
            x = x.to(device); y = y.to(device); m = m.to(device)
            optim.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=torch.bfloat16, enabled=device.type == 'cuda'):
                pred = model(past_values=x).regression_outputs.squeeze(-1)
                if pred.shape != y.shape:
                    pred = pred.view(y.shape)
                loss = ((pred - y) ** 2 * m).sum() / m.sum().clamp(min=1.0)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, ft.grad_clip)
            optim.step()
            running += float(loss.detach().item())
        print(f'  epoch {epoch:02d} | loss {running / max(1, len(train_loader)):.4f}')
    return model

def evaluate(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    sse = np.zeros(len(RUL_COLS), dtype=np.float64)
    n = np.zeros(len(RUL_COLS), dtype=np.float64)
    abs_err = np.zeros(len(RUL_COLS), dtype=np.float64)
    with torch.no_grad():
        for x, y, m in loader:
            x = x.to(device); y = y.to(device); m = m.to(device)
            pred = model(past_values=x).regression_outputs.squeeze(-1)
            if pred.shape != y.shape:
                pred = pred.view(y.shape)
            err = (pred - y).cpu().numpy() * RUL_CLIP
            mask = m.cpu().numpy()
            sse += (err ** 2 * mask).sum(axis=0)
            abs_err += (np.abs(err) * mask).sum(axis=0)
            n += mask.sum(axis=0)
    rmse = np.sqrt(sse / np.maximum(n, 1.0))
    mae = abs_err / np.maximum(n, 1.0)
    return {
        'rmse_per_component': dict(zip(RUL_COLS, rmse.tolist())),
        'mae_per_component': dict(zip(RUL_COLS, mae.tolist())),
        'rmse_mean': float(rmse.mean()),
        'mae_mean': float(mae.mean()),
    }

In [ ]:
N_DAYS = int(feat_fleet['day'].max() + 1)
FOLDS = expanding_window_folds(N_DAYS, n_folds=4, min_train_days=1800, val_days=400)
print('Folds:', [(len(t), len(v)) for t, v in FOLDS])

VARIANTS = [
    # (name, load_pretrained, freeze_encoder)
    ('scratch_all',        False, False),
    ('pretrained_all',     True,  False),
    ('pretrained_frozen',  True,  True),
]
cv_results = {name: [] for name, _, _ in VARIANTS}
for variant, load_pre, freeze in VARIANTS:
    print(f'
=== variant: {variant} (load_pre={load_pre}, freeze={freeze}) ===')
    for fold_idx, (train_range, val_range) in enumerate(FOLDS):
        print(f'-- fold {fold_idx}: train {train_range.start}..{train_range.stop} val {val_range.start}..{val_range.stop} --')
        train_ds = RULDataset(feat_fleet, TRAIN_PRINTERS, train_range,
                              feature_cols, RUL_COLS, CONTEXT_LENGTH, channel_mean, channel_std)
        val_ds = RULDataset(feat_fleet, VAL_PRINTERS, val_range,
                            feature_cols, RUL_COLS, CONTEXT_LENGTH, channel_mean, channel_std)
        if len(train_ds) == 0 or len(val_ds) == 0:
            print('  empty split, skipping fold')
            continue
        train_loader = DataLoader(train_ds, batch_size=ft.batch_size, shuffle=True, drop_last=True, num_workers=4, pin_memory=True, persistent_workers=True)
        val_loader = DataLoader(val_ds, batch_size=ft.batch_size, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)
        model = build_regression_model(load_pretrained=load_pre)
        train_one_fold(model, train_loader, freeze_encoder=freeze)
        metrics = evaluate(model, val_loader)
        metrics['fold'] = fold_idx
        cv_results[variant].append(metrics)
        print('  val', {k: round(v, 3) if isinstance(v, float) else v for k, v in metrics.items() if not isinstance(v, dict)})
with (RESULTS_DIR / 'cv_metrics.json').open('w', encoding='utf-8') as handle:
    json.dump(cv_results, handle, indent=2)

Folds: [(1800, 400), (2284, 400), (2768, 400), (3252, 400)]

=== variant: ssl_frozen ===
-- fold 0: train 0..1800 val 1800..2200 --


[transformers] Setting `do_mask_input` parameter to False.


load_pretrained=True: missing=2 unexpected=0
  epoch 00 | loss 0.0113
  epoch 01 | loss 0.0046
  epoch 02 | loss 0.0040
  epoch 03 | loss 0.0036
  epoch 04 | loss 0.0033
  epoch 05 | loss 0.0032
  val {'rmse_mean': 17.273, 'mae_mean': 14.045, 'fold': 0}
-- fold 1: train 0..2284 val 2284..2684 --


[transformers] Setting `do_mask_input` parameter to False.


load_pretrained=True: missing=2 unexpected=0
  epoch 00 | loss 0.0103
  epoch 01 | loss 0.0048
  epoch 02 | loss 0.0043
  epoch 03 | loss 0.0039
  epoch 04 | loss 0.0036
  epoch 05 | loss 0.0035
  val {'rmse_mean': 17.15, 'mae_mean': 14.054, 'fold': 1}
-- fold 2: train 0..2768 val 2768..3168 --


...

load_pretrained=True: missing=2 unexpected=0
  epoch 00 | loss 0.0092
  epoch 01 | loss 0.0049
  epoch 02 | loss 0.0043
  epoch 03 | loss 0.0040
  epoch 04 | loss 0.0039
  epoch 05 | loss 0.0037
  val {'rmse_mean': 20.641, 'mae_mean': 16.043, 'fold': 2}
-- fold 3: train 0..3252 val 3252..3652 --


[transformers] Setting `do_mask_input` parameter to False.


load_pretrained=True: missing=2 unexpected=0
  epoch 00 | loss 0.0090
  epoch 01 | loss 0.0051
  epoch 02 | loss 0.0045
  epoch 03 | loss 0.0042
  epoch 04 | loss 0.0040
  epoch 05 | loss 0.0039


[transformers] Setting `do_mask_input` parameter to False.


  val {'rmse_mean': 6.264, 'mae_mean': 5.04, 'fold': 3}

=== variant: scratch ===
-- fold 0: train 0..1800 val 1800..2200 --
  epoch 00 | loss 0.0089
  epoch 01 | loss 0.0014
  epoch 02 | loss 0.0012
  epoch 03 | loss 0.0012
  epoch 04 | loss 0.0011


In [ ]:
test_metrics: dict[str, dict] = {}
for variant, load_pre, freeze in VARIANTS:
    print(f'
=== test eval: {variant} ===')
    train_range = range(0, N_DAYS - 365)
    test_range = range(N_DAYS - 365, N_DAYS)
    train_ds = RULDataset(feat_fleet, TRAIN_PRINTERS, train_range,
                          feature_cols, RUL_COLS, CONTEXT_LENGTH, channel_mean, channel_std)
    test_ds = RULDataset(feat_fleet, TEST_PRINTERS, test_range,
                         feature_cols, RUL_COLS, CONTEXT_LENGTH, channel_mean, channel_std)
    train_loader = DataLoader(train_ds, batch_size=ft.batch_size, shuffle=True, drop_last=True, num_workers=4, pin_memory=True, persistent_workers=True)
    test_loader = DataLoader(test_ds, batch_size=ft.batch_size, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)
    model = build_regression_model(load_pretrained=load_pre)
    train_one_fold(model, train_loader, freeze_encoder=freeze)
    test_metrics[variant] = evaluate(model, test_loader)
    if variant == 'pretrained_frozen':
        # keep filename stable so 03_surrogate_search.ipynb keeps working
        torch.save(model.state_dict(), MODELS_DIR / 'rul_head_ssl.pt')
with (RESULTS_DIR / 'test_metrics.json').open('w', encoding='utf-8') as handle:
    json.dump(test_metrics, handle, indent=2)
test_metrics

In [ ]:
summary = pd.DataFrame(
    {
        variant: {'mae_mean': metrics['mae_mean'], 'rmse_mean': metrics['rmse_mean']}
        for variant, metrics in test_metrics.items()
    }
).T
print(summary.round(3))

if {'scratch_all', 'pretrained_frozen'}.issubset(test_metrics):
    same_regime = (summary.loc['scratch_all'] - summary.loc['pretrained_frozen']) / summary.loc['scratch_all'] * 100
    print('
SSL transfer signal (scratch_all vs pretrained_frozen, same head training regime; larger = better):')
    print(same_regime.round(2))

if {'pretrained_all', 'scratch_all'}.issubset(test_metrics):
    upper = (summary.loc['scratch_all'] - summary.loc['pretrained_all']) / summary.loc['scratch_all'] * 100
    print('
Upper bound (full fine-tune from SSL init vs scratch):')
    print(upper.round(2))